# 02. Retrieval (BM25) + RAG (Llama 3.2 3B)

| Langkah | Isi |
|---|---|
| 0. Setup | pengaturan model & cek Ollama |
| 1. Load data | corpus & ground truth dari notebook 01 |
| 2. BM25 | index leksikal untuk 3 corpus |
| 3. Fungsi pencarian | `search(corpus, query, k)` |
| 4. Evaluasi retrieval | Hit/Precision/Recall/MRR @1, @3, @5, @10 untuk 3 tombol |
| 5. RAG | prompt + Llama 3.2 3B, fungsi `tanya()` |
| 6. Evaluasi RAG singkat | validitas sitasi + simpan jawaban untuk dicek manual |

Retriever dense (embedding) menyusul nanti; untuk sekarang hanya BM25.

**Sebelum menjalankan:** nyalakan Ollama, dan pastikan model sudah ada: `ollama pull llama3.2:3b`.

## 0. Setup

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import ollama
from nltk.stem import PorterStemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tqdm.auto import tqdm

BASE = Path(r"E:\Tugas Semester 7\NLP\chatbot\final_project_2")                 # folder final_project_2
DATA = BASE / "data"
OUT  = BASE / "outputs"           # hasil evaluasi
OUT.mkdir(exist_ok=True)

LLM_MODEL = "llama3.2:3b"         # model LLM (lewat Ollama)
KS        = [1, 3, 5, 10]         # nilai K untuk evaluasi

# cek model di Ollama
available = [m.model for m in ollama.list().models]
status = "siap" if any(a.startswith(LLM_MODEL) for a in available) else f"BELUM ADA -> jalankan: ollama pull {LLM_MODEL}"
print(f"{LLM_MODEL}: {status}")

e:\Tugas Semester 7\NLP\chatbot\chatbot_nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


llama3.2:3b: siap


## 1. Load data

Hasil notebook 01. Teks yang dipakai untuk retrieval:

| Corpus | Teks yang dicari |
|---|---|
| islamqa | `judul + pertanyaan` (jawaban tidak ikut, karena yang dicari pertanyaan yang mirip) |
| quran | terjemahan Inggris `text_en` |
| hadith | terjemahan Inggris `text_en` |

In [2]:
islamqa = pd.read_csv(DATA / "islamqa.csv", dtype={"doc_id": str})
quran   = pd.read_csv(DATA / "quran.csv")
hadith  = pd.read_csv(DATA / "hadith.csv")

qrels_question = pd.read_csv(DATA / "qrels_question.csv", dtype=str)
qrels_quran    = pd.read_csv(DATA / "qrels_quran.csv", dtype=str)
qrels_hadith   = pd.read_csv(DATA / "qrels_hadith.csv", dtype={"query_id": str, "doc_id": str})

CORPUS = {
    "islamqa": {"ids": islamqa["doc_id"].tolist(), "text": (islamqa["judul"] + ". " + islamqa["pertanyaan"]).tolist()},
    "quran":   {"ids": quran["doc_id"].tolist(),   "text": quran["text_en"].tolist()},
    "hadith":  {"ids": hadith["doc_id"].tolist(),  "text": hadith["text_en"].tolist()},
}
for name, c in CORPUS.items():
    print(f"{name:<8} {len(c['ids']):>6,} dokumen")

islamqa   2,065 dokumen
quran     6,236 dokumen
hadith   26,725 dokumen


## 2. BM25 (leksikal)

Preprocessing: huruf kecil → ambil kata `[a-z0-9]+` → buang stopword & kata < 2 huruf → stemming (Porter).
BM25 memberi skor tinggi pada dokumen yang memuat kata-kata query, terutama kata yang jarang.

In [3]:
stemmer = PorterStemmer()

def preprocess(text):
    words = re.findall(r"[a-z0-9]+", str(text).lower())
    return [stemmer.stem(w) for w in words if len(w) >= 2 and w not in ENGLISH_STOP_WORDS]

bm25 = {}
for name, c in CORPUS.items():
    tokens = [preprocess(t) for t in tqdm(c["text"], desc=f"BM25 {name}")]
    bm25[name] = BM25Okapi(tokens)

BM25 hadith: 100%|██████████| 26725/26725 [00:38<00:00, 701.74it/s] 


## 3. Fungsi pencarian

`search(corpus, query, k)` mengembalikan `k` dokumen teratas: `[(doc_id, skor), ...]`.

In [4]:
def search(corpus, query, k=5):
    scores = bm25[corpus].get_scores(preprocess(query))
    idx = np.argsort(-scores)[:k]
    ids = CORPUS[corpus]["ids"]
    return [(ids[i], float(scores[i])) for i in idx]

# coba
quran_by_id = quran.set_index("doc_id")
for doc_id, score in search("quran", "Is it allowed to backbite someone?", k=3):
    print(f"{doc_id:<7} {score:.3f}  {quran_by_id.loc[doc_id, 'text_en'][:90]}")

26:211  9.049  It is not allowable for them, nor would they be able.
43:75   8.375  It will not be allowed to subside for them, and they, therein, are in despair.
86:17   7.289  So allow time for the disbelievers. Leave them awhile.


## 4. Evaluasi retrieval

Metrik untuk setiap query, lalu dirata-rata (K = 1, 3, 5, 10):

| Metrik | Arti |
|---|---|
| **Hit@K** | ada minimal 1 dokumen benar di top-K |
| **Precision@K** | dari K dokumen yang diambil, berapa yang benar |
| **Recall@K** | dari semua dokumen benar, berapa yang ditemukan di top-K |
| **MRR** | 1 / peringkat dokumen benar pertama (dalam top-10) |

Query & ground truth:
- **Pertanyaan**: 50 soal (asli & parafrase), 1 ground truth per soal. Precision@K maksimal 1/K, jadi yang utama Hit@K & MRR
- **Ayat**: pertanyaan islamqa (judul + pertanyaan) → ayat yang dikutip di jawabannya
- **Hadis**: sama, dilaporkan untuk **semua** ground truth dan yang **terverifikasi**

In [5]:
def evaluate(corpus, queries, relevant_sets):
    rows = []
    for query, relevant in tqdm(list(zip(queries, relevant_sets)), desc=corpus, leave=False):
        ranked = [d for d, _ in search(corpus, query, k=max(KS))]
        row = {}
        for k in KS:
            hits = sum(d in relevant for d in ranked[:k])
            row[f"Hit@{k}"] = float(hits > 0)
            row[f"P@{k}"] = hits / k
            row[f"R@{k}"] = hits / len(relevant)
        first = next((i + 1 for i, d in enumerate(ranked) if d in relevant), None)
        row["MRR"] = 1 / first if first else 0.0
        rows.append(row)
    return pd.DataFrame(rows).mean()

In [6]:
# ---- Retrieve Pertanyaan (islamqa), dipisah soal asli dan parafrase
results = {}
for jenis in ["asli", "parafrase"]:
    sub = qrels_question[qrels_question["jenis"] == jenis]
    results[f"pertanyaan ({jenis})"] = evaluate("islamqa", sub["query"].tolist(), [{d} for d in sub["doc_id"]])

cols = ["Hit@1", "Hit@3", "Hit@5", "Hit@10", "MRR"]
pd.DataFrame(results).T[cols].round(3)

,Hit@1,Hit@3,Hit@5,Hit@10,MRR
pertanyaan (asli),0.96,1.00,1.00,1.00,0.980
pertanyaan (parafrase),0.32,0.48,0.52,0.56,0.397


In [7]:
# ---- Retrieve Ayat & Hadis: query = judul + pertanyaan islamqa
query_text = (islamqa["judul"] + ". " + islamqa["pertanyaan"]).set_axis(islamqa["doc_id"])

def dalil_eval(corpus, qrels):
    gt = qrels.groupby("query_id")["doc_id"].apply(set)
    return evaluate(corpus, query_text[gt.index].tolist(), gt.tolist())

results["ayat"] = dalil_eval("quran", qrels_quran)
results["hadis (semua)"] = dalil_eval("hadith", qrels_hadith)
results["hadis (terverifikasi)"] = dalil_eval("hadith", qrels_hadith[qrels_hadith["terverifikasi"]])

cols = [f"{m}@{k}" for m in ["P", "R"] for k in KS] + ["Hit@5", "MRR"]
pd.DataFrame(results).T.loc[["ayat", "hadis (semua)", "hadis (terverifikasi)"], cols].round(3)

,P@1,P@3,P@5,P@10,R@1,R@3,R@5,R@10,Hit@5,MRR
ayat,0.050,0.030,0.023,0.017,0.021,0.038,0.048,0.068,0.100,0.073
hadis (semua),0.036,0.024,0.019,0.013,0.021,0.041,0.055,0.074,0.088,0.059
hadis (terverifikasi),0.038,0.027,0.021,0.014,0.025,0.049,0.067,0.091,0.097,0.064


In [8]:
# simpan semua hasil evaluasi retrieval
eval_bm25 = pd.DataFrame(results).T.round(4)
eval_bm25.index.name = "evaluasi"
eval_bm25.to_csv(OUT / "eval_retrieval_bm25.csv")
print("Tersimpan:", OUT / "eval_retrieval_bm25.csv")

Tersimpan: E:\Tugas Semester 7\NLP\chatbot\final_project_2\outputs\eval_retrieval_bm25.csv


## 5. RAG dengan Llama 3.2 3B

Alur `tanya(pertanyaan)`:
1. Cari **3 pertanyaan islamqa**, **3 ayat**, dan **3 hadis** paling relevan dengan BM25
2. Susun prompt: instruksi + sumber-sumber (dengan ID) + pertanyaan
3. LLM menulis jawaban ringkas **hanya dari sumber**, dengan sitasi seperti `[QA-12345]`, `[QS 2:275]`, `[bukhari:6018]`
4. Tampilkan hasil retrieval (ayat Arab + terjemahan, hadis, pertanyaan islamqa) dan jawaban LLM

Jawaban islamqa dipotong ±150 kata per dokumen agar prompt tetap muat di konteks LLM (4096 token).

In [9]:
N_SOURCES = 3         # jumlah sumber per jenis

SYSTEM_PROMPT = (
    "You are an assistant that answers questions about Islam using ONLY the sources provided: "
    "Q&A from islamqa.info, Quran verses, and hadith. Do not use outside knowledge and do not issue your own fatwa. "
    "Cite every statement with the source ID in square brackets, e.g. [QA-12345], [QS 2:275], [bukhari:6018]. "
    "Only cite IDs that appear in the sources. If the sources do not answer the question, say so. "
    "End with a short reminder to consult a qualified scholar."
)

USER_TEMPLATE = """Question: {question}

Sources:
{sources}

Write a concise answer (3-6 sentences) based only on the sources above, with citations."""

islamqa_by_id = islamqa.set_index("doc_id")
hadith_by_id  = hadith.set_index("doc_id")

def shorten(text, n_words=150):
    words = str(text).split()
    return " ".join(words[:n_words]) + (" ..." if len(words) > n_words else "")

def retrieve_all(question, k=N_SOURCES):
    return {name: search(name, question, k) for name in CORPUS}

def build_prompt(question, results):
    lines = []
    for doc_id, _ in results["islamqa"]:
        r = islamqa_by_id.loc[doc_id]
        lines.append(f"[QA-{doc_id}] Question: {r['judul']}\nAnswer: {shorten(r['jawaban'])}")
    for doc_id, _ in results["quran"]:
        lines.append(f"[QS {doc_id}] {quran_by_id.loc[doc_id, 'text_en']}")
    for doc_id, _ in results["hadith"]:
        lines.append(f"[{doc_id}] {shorten(hadith_by_id.loc[doc_id, 'text_en'], 100)}")
    return USER_TEMPLATE.format(question=question, sources="\n\n".join(lines))

def generate(prompt):
    response = ollama.chat(model=LLM_MODEL,
                           messages=[{"role": "system", "content": SYSTEM_PROMPT},
                                     {"role": "user", "content": prompt}],
                           options={"temperature": 0.1, "num_ctx": 4096, "num_predict": 500})
    return response["message"]["content"].strip()

In [10]:
def tanya(question, show=True):
    """Satu fungsi: retrieval 3 corpus -> LLM menjawab -> tampilkan semuanya."""
    results = retrieve_all(question)
    start = time.time()
    answer = generate(build_prompt(question, results))
    llm_time = time.time() - start
    if not show:
        return {"question": question, "results": results, "answer": answer, "llm_time": llm_time}

    print("PERTANYAAN:", question)
    print("\n================ JAWABAN LLM ================")
    print(answer)
    print(f"\n(LLM {LLM_MODEL}: {llm_time:.1f} detik | retriever: BM25)")

    print("\n================ RETRIEVE AL-QUR'AN ================")
    for doc_id, score in results["quran"]:
        r = quran_by_id.loc[doc_id]
        print(f"\nQS. {r['surah_name']} {doc_id}   (skor {score:.3f})\n{r['text_ar']}\n\"{r['text_en']}\"")

    print("\n================ RETRIEVE HADIS ================")
    for doc_id, score in results["hadith"]:
        r = hadith_by_id.loc[doc_id]
        print(f"\n{doc_id}   (skor {score:.3f})\n{shorten(r['text_ar'], 40)}\n\"{shorten(r['text_en'], 80)}\"")

    print("\n================ RETRIEVE PERTANYAAN (islamqa) ================")
    for doc_id, score in results["islamqa"]:
        r = islamqa_by_id.loc[doc_id]
        print(f"\n[{doc_id}] {r['judul']}   (skor {score:.3f})\n{r['url']}\nJawaban: {shorten(r['jawaban'], 60)}")

In [11]:
tanya("Is it permissible to backbite someone if what I say is true?")

PERTANYAAN: Is it permissible to backbite someone if what I say is true?

================ JAWABAN LLM ================
Based on the sources provided, it is not permissible to backbite someone, even if what is said is true. [QA-184850] states that insulting people and mentioning their faults is considered bad behavior that can lead to resentment and hatred. [QA-106417] mentions that backbiting is permitted in certain situations, such as complaining to the ruler or judge, but this does not apply to general conversation. [QA-105391] also mentions that backbiting is permitted in specific situations, but this does not include general conversation. [QS 3:166] and [QS 51:5] emphasize the importance of truth and certainty, which is not guaranteed in backbiting. [QA-184850] and [abudawud:4874] also state that if what is said about someone is true, it is considered slander, and if not, it is considered reviling.

(LLM llama3.2:3b: 24.9 detik | retriever: BM25)

================ RETRIEVE AL-QUR'

In [12]:
# Coba sendiri: ganti pertanyaannya
tanya("How should a Muslim treat his neighbours?")

PERTANYAAN: How should a Muslim treat his neighbours?

================ JAWABAN LLM ================
A Muslim should treat his neighbors with kindness and respect, as emphasized in the Quran and Sunnah. Allah commands Muslims to be good to their neighbors, including those who are Muslim and those who are not [QS 4:36]. The Prophet (peace and blessings of Allah be upon him) also instructed believers to treat their neighbors well, honor their guests, and show good manners [ibn Majah:3672]. Additionally, a neighbor has the best claim to the house or land of the neighbor, and a house that treats orphans well is considered the best among Muslims [ibn Majah:3679]. However, the extent of these rights may vary depending on the situation and the type of neighbor.

(LLM llama3.2:3b: 10.6 detik | retriever: BM25)

================ RETRIEVE AL-QUR'AN ================

QS. Al-Qalam 68:35   (skor 15.931)
أَفَنَجْعَلُ ٱلْمُسْلِمِينَ كَٱلْمُجْرِمِينَ
"Then will We treat the Muslims like the criminals?

## 6. Evaluasi RAG singkat

Dijalankan pada beberapa soal **parafrase** (gaya pertanyaan pengguna). Yang dihitung otomatis:
- **kunci_islamqa_ditemukan**: apakah pertanyaan islamqa kuncinya ikut masuk ke sumber LLM
- **citation validity**: proporsi sitasi yang memang ada di sumber yang diberikan (bukan ID karangan LLM)
- **waktu LLM**

Semua jawaban disimpan ke `outputs/rag_bm25_llama32_3b.csv` untuk dinilai manual bersama
(benar/sebagian/salah, ada halusinasi atau tidak).

In [13]:
N_EVAL = 10
eval_questions = qrels_question[qrels_question["jenis"] == "parafrase"].head(N_EVAL)

CITE = re.compile(r"\[(QA-\d+|QS \d+:\d+|[a-z]+:\d+)\]")

rows = []
for _, q in tqdm(eval_questions.iterrows(), total=len(eval_questions), desc="RAG"):
    res = tanya(q["query"], show=False)
    given = {f"QA-{d}" for d, _ in res["results"]["islamqa"]} \
          | {f"QS {d}" for d, _ in res["results"]["quran"]} \
          | {d for d, _ in res["results"]["hadith"]}
    cited = set(CITE.findall(res["answer"]))
    rows.append({
        "query_id": q["query_id"], "question": q["query"],
        "kunci_islamqa_ditemukan": q["doc_id"] in {d for d, _ in res["results"]["islamqa"]},
        "jumlah_sitasi": len(cited),
        "citation_validity": len(cited & given) / len(cited) if cited else np.nan,
        "llm_detik": round(res["llm_time"], 1),
        "answer": res["answer"],
    })

rag_eval = pd.DataFrame(rows)
rag_eval.to_csv(OUT / "rag_bm25_llama32_3b.csv", index=False)
print(rag_eval[["kunci_islamqa_ditemukan", "jumlah_sitasi", "citation_validity", "llm_detik"]].mean().round(3).to_string())
rag_eval[["query_id", "question", "kunci_islamqa_ditemukan", "jumlah_sitasi", "citation_validity", "llm_detik"]]

RAG: 100%|██████████| 10/10 [02:06<00:00, 12.69s/it]

kunci_islamqa_ditemukan     0.60
jumlah_sitasi               2.10
citation_validity           1.00
llm_detik                  12.59


,query_id,question,kunci_islamqa_ditemukan,jumlah_sitasi,citation_validity,llm_detik
0,Q26,am i allowed to count my dhikir with a counter...,True,3,1.0,12.7
1,Q27,what does envy mean in islam?,True,3,1.0,10.9
2,Q28,I secretly took caash from my parents in the p...,False,2,1.0,13.4
3,Q29,does a housewife get rewarded for doing househ...,False,2,1.0,12.5
4,Q30,if someone left islam after a tragedy that str...,False,2,1.0,12.7
5,Q31,can muslims use a string of beads to count the...,True,3,1.0,12.1
6,Q32,is it okay to say the remembrances more times ...,True,3,1.0,14.9
7,Q33,should i lift my hands while saying the mornin...,True,1,1.0,14.4
8,Q34,is it wrong to say that time betrays people?,False,2,1.0,11.0
9,Q35,what rewards does islam promise to people who ...,True,0,NaN,11.3


In [14]:
# Jawaban lengkap untuk dinilai manual
for _, r in rag_eval.iterrows():
    print(f"=== {r['query_id']}: {r['question']}\n{r['answer']}\n")

=== Q26: am i allowed to count my dhikir with a counter ring worn on my finger?
Based on the sources provided, it appears that counting dhikr on one's fingers is a Sunnah, as described in [QA-152672] and [QA-198080]. However, there is no specific ruling on whether a counter ring worn on the finger is permissible for counting dhikr. The only relevant hadith mentioned is [bukhari:5873], where Allah's Messenger (blessings and peace of Allah be upon him) wore a ring on his hand, but this does not explicitly address the use of a counter ring. [QA-198080] mentions that the Sunnah is for the Muslim to count tasbeeh and tahleel on their fingers, but does not provide a clear answer on the use of a counter ring. Therefore, it is unclear whether a counter ring is permissible for counting dhikr.

=== Q27: what does envy mean in islam?
In Islam, envy (hasad) refers to wishing that a blessing bestowed upon someone be taken away. It is considered a vile and base characteristic, and a major sin [QA-18